In [ ]:
import requests
import time
import json
from pprint import pprint

from py_clob_client.client import ClobClient
from py_clob_client.clob_types import OrderArgs, MarketOrderArgs, OrderType, OpenOrderParams, BalanceAllowanceParams, AssetType
from py_clob_client.order_builder.constants import BUY, SELL

GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API = "https://data-api.polymarket.com"
CLOB_API = "https://clob.polymarket.com"
def track_price(token_id, duration_seconds, interval):
    """Track price changes in real-time."""
    print(f"Tracking price for {duration_seconds}s...\n")

    client = ClobClient(CLOB_API)
    start_time = time.time()
    prices = []

    while time.time() - start_time < duration_seconds:
        mid = client.get_midpoint(token_id)
        mid_price = float(mid['mid'])
        timestamp = time.strftime("%H:%M:%S")
        prices.append(mid_price)

        change = ""
        if len(prices) > 1:
            diff = prices[-1] - prices[-2]
            change = f" ({'+' if diff >= 0 else ''}{diff:.4f})"

        print(f"[{timestamp}] Price: {mid_price}{change}")
        time.sleep(interval)

    print(f"\nTotal change: {prices[-1] - prices[0]:.4f}")
    return prices
prices = track_price(112493648051773944964837615042352399305043803435108343315031715517311229730536, duration_seconds=600, interval=60)

In [1]:
import requests
import time
from datetime import datetime
from ipynb.fs.full.AutoEmail import send_email

CLOB_URL = "https://clob.polymarket.com"
GAMMA_URL = "https://gamma-api.polymarket.com"

def get_token_price(token_id: str) -> float | None:
    """Fetch current mid-market price for a token."""
    try:
        resp = requests.get(f"{CLOB_URL}/midpoint", params={"token_id": token_id}, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        return float(data["mid"])
    except Exception as e:
        print(f"  [Error fetching price]: {e}")
        return None

def get_event_from_token(token_id: str) -> dict:
    """Look up event/market info from a token ID via Gamma."""
    
    resp = requests.get(
        f"{GAMMA_URL}/markets",
        params={"clob_token_ids": token_id},
        timeout=5
    )
    resp.raise_for_status()
    markets = resp.json()

    if not markets:
        raise ValueError(f"No market found for token_id: {token_id}")

    market = markets[0]
    return market.get("question")

def monitor_price(
    token_id: str,
    target_price: float,
    direction: str = "above",   # "above" or "below"
    poll_interval: int = 5,     # seconds between checks
    on_trigger=None             # optional callback function
):
    """
    Polls a Polymarket token price until it hits the target.

    Args:
        token_id:      The Polymarket token/outcome ID
        target_price:  Price threshold (0.0 to 1.0)
        direction:     Trigger when price goes 'above' or 'below' target
        poll_interval: Seconds between each price check
        on_trigger:    Optional function to call when target is hit
    """
    print(f"Monitoring token: {token_id}")
    print(f"Waiting for price to go {direction} {target_price:.4f}")
    print(f"Polling every {poll_interval}s...\n")

    while True:
        price = get_token_price(token_id)
        now = datetime.now().strftime("%H:%M:%S")

        if price is not None:
            print(f"[{now}] Current price: {price:.4f}", end="")

            triggered = (
                (direction == "above" and price >= target_price) or
                (direction == "below" and price <= target_price)
            )

            if triggered:
                print(f"  ✅ TARGET HIT! Price {price:.4f} is {direction} {target_price:.4f}")
                if on_trigger:
                    on_trigger(token_id, price, target_price)
                return price
            else:
                print(f"  (target: {direction} {target_price:.4f})")
        
        time.sleep(poll_interval)


# --- Example callback when target is hit ---
def on_price_hit(token_id, current_price, target_price):
    print(f"\n🔔 ALERT: Token {token_id}")
    print(f"   Reached {current_price:.4f} (target was {target_price:.4f})")
    send_email(subject="Polymarket Alert", body=f"Token {get_event_from_token(token_id)} hit target price: {current_price:.4f}", to_email="arvindbijulal@gmail.com")
    # e.g. place an order, send a notification, etc.


if __name__ == "__main__":
    TOKEN_ID     = 6628882303864594731548894308977075373030062856864705920016694126013551708713 # paste your token ID
    TARGET_PRICE = 0.6                  # e.g. 75% probability
    DIRECTION    = "above"               # "above" or "below"
    POLL_EVERY   = 60                     # seconds

    monitor_price(
        token_id=TOKEN_ID,
        target_price=TARGET_PRICE,
        direction=DIRECTION,
        poll_interval=POLL_EVERY,
        on_trigger=on_price_hit
    )

Monitoring token: 6628882303864594731548894308977075373030062856864705920016694126013551708713
Waiting for price to go above 0.6000
Polling every 60s...

[22:14:26] Current price: 0.7050  ✅ TARGET HIT! Price 0.7050 is above 0.6000

🔔 ALERT: Token 6628882303864594731548894308977075373030062856864705920016694126013551708713
   Reached 0.7050 (target was 0.6000)


In [9]:
import requests
import time
import json
from pprint import pprint

from py_clob_client.client import ClobClient
from py_clob_client.clob_types import OrderArgs, MarketOrderArgs, OrderType, OpenOrderParams, BalanceAllowanceParams, AssetType
from py_clob_client.order_builder.constants import BUY, SELL

GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API = "https://data-api.polymarket.com"
CLOB_API = "https://clob.polymarket.com"


def get_token_id(slug):
    response = requests.get(
        f"https://gamma-api.polymarket.com/markets?slug={slug}",
        params={"active": "true", "closed": "false", "limit": 1}
    )
    markets = response.json()
    market = markets[0]
    return json.loads(market['clobTokenIds'])[0]
get_token_id("cl-up-or-down-on-march-11-2026")

'58701466512653623009277862062220316069590561202991636378642856811587452042517'

In [ ]:
import requests

GAMMA_URL = "https://gamma-api.polymarket.com"

def get_event_from_token(token_id: str) -> dict:
    """Look up event/market info from a token ID via Gamma."""
    
    resp = requests.get(
        f"{GAMMA_URL}/markets",
        params={"clob_token_ids": token_id},
        timeout=5
    )
    resp.raise_for_status()
    markets = resp.json()

    if not markets:
        raise ValueError(f"No market found for token_id: {token_id}")

    market = markets[0]
    return market.get("question")


if __name__ == "__main__":
    TOKEN_ID = "8006852724177121783175530504461885300457337321145618842924488550959073671809"

    info = get_event_from_token(TOKEN_ID)
    print(info)